# Nền tảng 1 — Từ `loss` trong log đến chỉ số bpc

Notebook này bắt đầu từ một con số bạn đã nhìn hàng chục lần trong `train.log`, rồi đi từng bước tới `bpc` —
chỉ số mà toàn bộ kết quả của project được phát biểu bằng nó.

Mỗi khái niệm đều có một cell chạy được. Cứ chạy, đổi số, xem nó gãy ở đâu.

**Chạy bằng kernel pixi của project** (`.pixi/envs/default/bin/python`).

In [1]:
import json
import math
import unicodedata
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:   # tìm gốc repo, chạy được từ mọi thư mục
    ROOT = ROOT.parent
DATA = ROOT / "kaggle" / "outputs" / "vitok-data"
RUNS = ROOT / "kaggle" / "outputs"
print("project:", ROOT)
print("có dữ liệu:", DATA.exists(), "| có kết quả:", (RUNS / "results-v6").exists())

project: /home/geminitt/class/deep-learning/project
có dữ liệu: True | có kết quả: True


## 1. Con số xuất phát

Dòng cuối trong log huấn luyện `bpe-nfc` ở d8. Bạn đã biết `loss` là giá trị `F.cross_entropy` trả về.

In [2]:
log = RUNS / "results-v6" / "runs" / "bpe-nfc_d8_s0" / "train.log"
steps = [l for l in log.read_text().splitlines() if l.startswith("step ")]
print(steps[-1])

loss_nat = float(steps[-1].split("loss:")[1].split("|")[0])
print("\nloss =", loss_nat, "nat mỗi token")

step 07628/07629 (99.99%) | loss: 2.615051 | lrm: 0.05 | dt: 1335.94ms | tok/sec: 49,055 | bf16_mfu: 0.00 | epoch: 1 pq: 8 rg: 15 | total time: 168.73m | eta: 0.0m

loss = 2.615051 nat mỗi token


## 2. Nat và bit

`F.cross_entropy` dùng logarit tự nhiên, nên đơn vị của nó là **nat**. Nếu dùng logarit cơ số 2 thì đơn vị là
**bit**. Hai đơn vị chỉ khác nhau một hệ số:

$$1\ \text{nat} = \frac{1}{\ln 2} \approx 1{,}4427\ \text{bit}$$

**Ký hiệu mới:** $\ln 2 \approx 0{,}693$ — hệ số đổi giữa hai loại logarit: $\log_2 u = \ln u / \ln 2$.

Vì sao thư viện học sâu chọn nat: đạo hàm của $\ln$ gọn hơn, không kéo theo hệ số $1/\ln 2$ ở mọi bước lan
truyền ngược.

In [3]:
loss_bit = loss_nat / math.log(2)
print(f"{loss_nat:.3f} nat = {loss_bit:.3f} bit  (mỗi token)")
print(f"kiểm tra ngược: {loss_bit:.3f} bit = {loss_bit * math.log(2):.3f} nat")

2.615 nat = 3.773 bit  (mỗi token)
kiểm tra ngược: 3.773 bit = 2.615 nat


## 3. "3,77 bit" nghĩa đen là gì

Nghĩa đen: nếu nén văn bản bằng chính model này, trung bình mỗi token tốn 3,77 bit để lưu.

Đây không phải cách nói ví von. Mục 5 sẽ dẫn ra định lý cho phép nói như vậy. Trước hết cần hiểu vì sao một sự
kiện có xác suất $p$ lại mang $-\log p$ đơn vị thông tin.

Ta muốn một hàm $I(p)$ đo "lượng thông tin nhận được khi biết sự kiện xác suất $p$ đã xảy ra", thoả ba yêu cầu:

1. $I(1) = 0$ — biết một việc chắc chắn xảy ra thì không nhận thêm thông tin nào.
2. $I$ giảm khi $p$ tăng — việc càng bất ngờ càng nhiều thông tin.
3. $I(p_1 p_2) = I(p_1) + I(p_2)$ — hai sự kiện độc lập thì thông tin cộng lại. Ở đây $p_1$, $p_2$ là xác suất của\n   hai sự kiện độc lập, nên $p_1 p_2$ là xác suất để **cả hai** cùng xảy ra.

Yêu cầu 3 là điều kiện mạnh nhất: hàm liên tục duy nhất biến nhân thành cộng là logarit. Nên $I(p) = -c\log p$ (với $c > 0$ là một hằng số),
và hằng số $c$ chính là việc chọn đơn vị.

In [4]:
def info_bit(p):
    return -math.log2(p) + 0.0

for p in [1.0, 0.5, 0.25, 0.125, 0.05, 0.02]:
    print(f"p = {p:5.3f} -> {info_bit(p):6.3f} bit = {info_bit(p) * math.log(2):6.3f} nat")

# kiểm tra yêu cầu 3: thông tin của hai sự kiện độc lập cộng lại
p1, p2 = 0.25, 0.1
print(f"\nI({p1}*{p2}) = {info_bit(p1*p2):.4f}")
print(f"I({p1}) + I({p2}) = {info_bit(p1) + info_bit(p2):.4f}")

p = 1.000 ->  0.000 bit =  0.000 nat
p = 0.500 ->  1.000 bit =  0.693 nat
p = 0.250 ->  2.000 bit =  1.386 nat
p = 0.125 ->  3.000 bit =  2.079 nat
p = 0.050 ->  4.322 bit =  2.996 nat
p = 0.020 ->  5.644 bit =  3.912 nat

I(0.25*0.1) = 5.3219
I(0.25) + I(0.1) = 5.3219


Đổi `p1`, `p2` thành số khác và chạy lại: hai dòng cuối luôn bằng nhau. Đó là tính cộng tính, và nó buộc $I$
phải là logarit.

## 4. Entropy là kỳ vọng của thông tin

Nếu biến ngẫu nhiên $X$ có phân phối $p$, thì lượng thông tin là một biến ngẫu nhiên: nó bằng $-\log_2 p(x)$
khi $X = x$. **Entropy** là kỳ vọng của đại lượng đó — tức trung bình có trọng số, trọng số là chính $p$:

$$H(p) = \mathbb{E}_{x \sim p}\left[-\log_2 p(x)\right] = -\sum_x p(x)\log_2 p(x)$$

**Ký hiệu mới**
- $p(x)$ — xác suất để biến ngẫu nhiên nhận giá trị $x$
- $\mathbb{E}_{x \sim p}[\cdot]$ — kỳ vọng khi $x$ rút theo $p$, tức trung bình có trọng số $p(x)$
- $\sum_x$ — tổng qua mọi giá trị $x$ có thể có

In [5]:
def entropy_bit(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]                       # 0*log0 = 0 theo quy ước
    return float(-(p * np.log2(p)).sum())

p_lech = [0.5, 0.25, 0.125, 0.125]
p_deu = [0.25, 0.25, 0.25, 0.25]
print("phân phối lệch :", entropy_bit(p_lech), "bit")
print("phân phối đều  :", entropy_bit(p_deu), "bit   (= log2(4))")

phân phối lệch : 1.75 bit
phân phối đều  : 2.0 bit   (= log2(4))


Phân phối lệch có entropy nhỏ hơn: dễ đoán hơn thì cần ít thông tin hơn để mô tả.

Phân phối đều luôn có entropy **lớn nhất** trên cùng số khả năng: $H = \log_2 n$. Đây cho ta một mốc trên cho
project — một model chưa học được gì sẽ tốn đúng $\log_2(\text{vocab})$ bit mỗi token.

Chú ý vocab ở đây là **16.009** (16.000 token học được + 9 special token), không phải 16.064. nanochat đệm
`lm_head` lên 16.064 hàng cho nhanh, nhưng cắt 55 cột thừa khỏi logits trước khi tính loss
(`logits = logits[..., :self.config.vocab_size]` trong `gpt.py`), nên softmax chỉ trải trên 16.009 token.

In [6]:
vocab_that = 16009            # 16.000 token + 9 special; phần đệm tới 16.064 bị cắt khỏi logits
print(f"model đoán đều: {math.log2(vocab_that):.3f} bit mỗi token")
print(f"model của bạn : {loss_bit:.3f} bit mỗi token")
print(f"-> đã đi được {1 - loss_bit / math.log2(vocab_that):.1%} đường từ mốc đoán đều xuống 0")

model đoán đều: 13.967 bit mỗi token
model của bạn : 3.773 bit mỗi token
-> đã đi được 73.0% đường từ mốc đoán đều xuống 0


## 5. Vì sao entropy là "số bit tối thiểu"

Xét bài toán mã hoá: gán cho mỗi ký hiệu một dãy bit sao cho ghép lại vẫn giải mã được không nhập nhằng. Điều
kiện đủ thường dùng là **mã tiền tố**: không mã nào là tiền tố của mã khác.

**Định lý mã hoá nguồn của Shannon** nói hai chiều:

- Với mọi mã tiền tố, độ dài trung bình $L = \sum_x p(x)\ell(x) \ge H(p)$, với $\ell(x)$ là số bit của mã gán
  cho $x$. $L$ là một **hằng số** tính từ $p$ và bộ mã, không phụ thuộc mẫu nào.
- Tồn tại mã tiền tố với $L < H(p) + 1$.

Kiểm chứng bằng tay với phân phối lệch ở trên.

In [7]:
code = {"A": "0", "B": "10", "C": "110", "D": "111"}
probs = {"A": 0.5, "B": 0.25, "C": 0.125, "D": 0.125}

L = sum(probs[k] * len(v) for k, v in code.items())
print("L =", L, "bit")
print("H =", entropy_bit(list(probs.values())), "bit")

# mã tiền tố: không mã nào là tiền tố của mã khác
for a in code.values():
    for b in code.values():
        if a != b:
            assert not b.startswith(a), (a, b)
print("\nDone!")

L = 1.75 bit
H = 1.75 bit

Done!


$L = H$ đúng bằng nhau vì mọi xác suất ở đây là luỹ thừa của $1/2$ — trường hợp lý tưởng.

Thử đổi `xac_suat` thành `{"A": 0.7, "B": 0.1, "C": 0.1, "D": 0.1}` rồi chạy lại: khi đó $L > H$, vì mã dài
nguyên bit không khớp được với xác suất không phải luỹ thừa của $1/2$.

Kết luận cần nhớ: **"$H$ bit" nghĩa là "trung bình cần $H$ bit để lưu một mẫu, và không thể ít hơn"**. Model
ngôn ngữ tốt và bộ nén tốt là một bài toán.

In [8]:
# mô phỏng: sinh 200.000 ký hiệu theo phân phối, mã hoá thật, đo số bit thực tế
rng = np.random.default_rng(0)
ky_hieu = list(probs)
mau = rng.choice(ky_hieu, size=200_000, p=[probs[k] for k in ky_hieu])
tong_bit = sum(len(code[k]) for k in mau)
print(f"số bit thực tế / ký hiệu = {tong_bit / len(mau):.4f}")
print(f"entropy                  = {entropy_bit(list(probs.values())):.4f}")

số bit thực tế / ký hiệu = 1.7471
entropy                  = 1.7500


## 6. Cross-entropy: khi model chưa đúng

Trong thực tế ta không biết phân phối thật $p$. Ta có model $q$. Nếu thiết kế mã tối ưu cho $q$ (dài
$-\log_2 q(x)$ bit) rồi dùng nó cho dữ liệu ra từ $p$, độ dài trung bình là **cross-entropy**:

$$H(p, q) = -\sum_x p(x)\log_2 q(x)$$

**Ký hiệu mới:** $q(x)$ — xác suất **model** gán cho $x$ (còn $p(x)$ là xác suất thật, ta không biết).

Phần trả thêm so với mức tối thiểu chính là **KL divergence**:

$$H(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q), \qquad D_{\mathrm{KL}}(p \parallel q) = \sum_x p(x)\log_2\frac{p(x)}{q(x)} \ge 0$$

**Ký hiệu mới:** $D_{\mathrm{KL}}(p \parallel q)$ — số bit trả thêm vì dùng $q$ thay cho $p$. Thứ tự hai đối số quan
trọng: nói chung $D_{\mathrm{KL}}(p \parallel q) \ne D_{\mathrm{KL}}(q \parallel p)$.

In [9]:
def cross_entropy_bit(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    return float(-(p * np.log2(q)).sum())

def kl_bit(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    return float((p * np.log2(p / q)).sum())

p = np.array([0.5, 0.25, 0.125, 0.125])
q = np.array([0.4, 0.3, 0.2, 0.1])          # model lệch

print(f"H(p)      = {entropy_bit(p):.4f} bit   <- không thể tránh")
print(f"H(p,q)    = {cross_entropy_bit(p, q):.4f} bit   <- thực tế phải trả")
print(f"KL(p||q)  = {kl_bit(p, q):.4f} bit   <- phần phạt do model sai")
print(f"kiểm tra: H(p) + KL = {entropy_bit(p) + kl_bit(p, q):.4f}")

H(p)      = 1.7500 bit   <- không thể tránh
H(p,q)    = 1.8007 bit   <- thực tế phải trả
KL(p||q)  = 0.0507 bit   <- phần phạt do model sai
kiểm tra: H(p) + KL = 1.8007


Chứng minh $D_{\mathrm{KL}} \ge 0$, dùng bất đẳng thức $\ln u \le u - 1$ (đúng với mọi $u > 0$, dấu bằng khi $u = 1$):

$$-D_{\mathrm{KL}}(p \parallel q)\ln 2 = \sum_x p(x)\ln\frac{q(x)}{p(x)} \le \sum_x p(x)\left(\frac{q(x)}{p(x)} - 1\right) = \sum_x q(x) - \sum_x p(x) = 0$$

**Đọc từng bước**
1. Nhân với $\ln 2$ để đổi $\log_2$ sang $\ln$; lật $\frac{p}{q}$ thành $\frac{q}{p}$ để đổi dấu.
2. Dấu $\le$: áp $\ln u \le u - 1$ với $u = \frac{q(x)}{p(x)}$.
3. Bằng 0 vì $\sum_x q(x) = \sum_x p(x) = 1$.

nên $D_{\mathrm{KL}} \ge 0$, bằng 0 khi và chỉ khi $q = p$ tại mọi điểm. Kiểm chứng bằng mô phỏng:

In [ ]:
# KL >= 0 với mọi q: thử 10.000 model ngẫu nhiên, không cái nào cho KL âm
rng = np.random.default_rng(1)
kl_min = min(kl_bit(p, rng.dirichlet(np.ones(4))) for _ in range(10_000))
print(f"KL nhỏ nhất trong 10.000 lần thử: {kl_min:.6f}  (luôn >= 0, bằng 0 chỉ khi q = p)")
print(f"KL khi q = p: {kl_bit(p, p):.6f}")

KL nhỏ nhất trong 10.000 lần thử: 0.002483  (luôn >= 0, bằng 0 chỉ khi q = p)
KL khi q = p: 0.000000


Vì sao điều này quan trọng cho project: khi so `bpe-nfc` với `super-nfc` trên cùng dữ liệu, $H(p)$ giống nhau ở
cả hai, nên

$$H(p, q_A) - H(p, q_B) = D_{\mathrm{KL}}(p \parallel q_A) - D_{\mathrm{KL}}(p \parallel q_B)$$

**Ký hiệu mới:** $q_A$, $q_B$ — hai model đem so (ví dụ `super-nfc` và `bpe-nfc`).

Hiệu cross-entropy đúng bằng hiệu "độ lệch so với tiếng Việt thật". Không lẫn thành phần nào khác.

## 6b. Model sinh ra các xác suất đó bằng cách nào

Ở mỗi vị trí $t$, Transformer xuất ra một vector số thực $z^{(t)} \in \mathbb{R}^{|V|}$ gọi là **logits** —
trong code là kết quả của `lm_head`. Logits chưa phải xác suất: chúng có thể âm và không tổng thành 1. Phép
biến đổi sang xác suất là **softmax**:

$$q(x_t = j \mid x_{<t}) = \frac{\exp(z^{(t)}_j)}{\sum_{k=1}^{|V|} \exp(z^{(t)}_k)}$$

**Ký hiệu mới**
- $x_t$ — token ở vị trí $t$; $x_{<t}$ — mọi token đứng trước nó; dấu $\mid$ đọc là "với điều kiện"
- $|V|$ — số token trong vocab (16.009 ở project)
- $z^{(t)}_j$ — logit của token thứ $j$ trong vocab, tại vị trí $t$

Mẫu số là tổng trên toàn vocab, nên mọi token cạnh tranh nhau: tăng xác suất cho một token buộc phải giảm cho
các token còn lại.

Một chi tiết nữa: công thức điều kiện cần "phần đầu" để dựa vào, nhưng $x_1$ không có gì đứng trước. Cách xử lý
là thêm token **BOS** (`<|bos|>`) ở đầu. BOS là **điều kiện**, không phải **mục tiêu dự đoán** — model không
bao giờ phải đoán ra nó, nên nó không đóng góp nat nào. Trong `vitok/eval.py`, BOS nằm trong dãy đầu vào `x`,
còn dãy mục tiêu `y` bắt đầu từ $x_1$.

In [11]:
def softmax(z):
    z = np.asarray(z, float)
    e = np.exp(z - z.max())          # trừ max cho ổn định số học, không đổi kết quả
    return e / e.sum()

logits = np.array([2.0, 1.0, -1.0, 0.5])
q_vi_tri = softmax(logits)
print("logits  :", logits)
print("xác suất:", np.round(q_vi_tri, 4), "| tổng =", q_vi_tri.sum())
print("trừ max hay không, kết quả như nhau:", np.allclose(softmax(logits), softmax(logits - 100)))

# nếu token thật là token số 2 (xác suất thấp nhất), vị trí này tốn:
print(f"\ntoken thật = 2 -> {-math.log(q_vi_tri[2]):.3f} nat = {-math.log2(q_vi_tri[2]):.3f} bit")

logits  : [ 2.   1.  -1.   0.5]
xác suất: [0.6095 0.2242 0.0303 0.136 ] | tổng = 0.9999999999999999
trừ max hay không, kết quả như nhau: True

token thật = 2 -> 3.495 nat = 5.042 bit


## 7. Từ công thức sang một model chạy được

Công thức cross-entropy cần biết $p$. Ta không biết, nên thay bằng **phân phối thực nghiệm**: ở mỗi vị trí,
token thật có xác suất 1, mọi token khác xác suất 0. Tổng thu về một số hạng:

$$\ell_t = -\ln q(x_t \mid x_{<t})$$

**Ký hiệu mới:** $\ell_t$ — loss (nat) tại vị trí $t$, đúng giá trị `F.cross_entropy` tính cho vị trí đó.

Để thấy tận mắt, ta dựng một model thật — loại đơn giản nhất có thể: **unigram**, chỉ đếm tần suất token, bỏ qua
ngữ cảnh. Dùng chính tokenizer `bpe-nfc` 16k của project.

In [12]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(str(DATA / "tokenizers-16k" / "bpe-nfc" / "tokenizer.json"))
docs = [json.loads(l)["text"] for l in (DATA / "test.jsonl").read_text(encoding="utf-8").splitlines()]
train_docs, eval_docs = docs[:1500], docs[1500:1600]      # tách để không đánh giá trên dữ liệu đã đếm
print(f"{len(train_docs)} văn bản để đếm tần suất, {len(eval_docs)} văn bản để đánh giá")

V = tok.get_vocab_size()
dem = np.ones(V)                                          # cộng 1 để không có xác suất 0 (làm mượt Laplace)
for e in tok.encode_batch(train_docs, add_special_tokens=False):
    np.add.at(dem, e.ids, 1)
q_unigram = dem / dem.sum()
print(f"vocab = {V}, 5 token hay gặp nhất:",
      [tok.id_to_token(i) for i in np.argsort(-q_unigram)[:5]])

1500 văn bản để đếm tần suất, 100 văn bản để đánh giá
vocab = 16009, 5 token hay gặp nhất: [',', '.', 'Ġ', 'ĠvÃł', '.Ċ']


Bây giờ tính tổng nat của mỗi văn bản đánh giá, đúng như [`vitok.eval.sequence_nats`](../../../src/vitok/eval.py)
làm với model thật: cộng $-\ln q$ ở mọi vị trí.

In [13]:
ket_qua = []
for text, e in zip(eval_docs, tok.encode_batch(eval_docs, add_special_tokens=False)):
    nats = float(-np.log(q_unigram[e.ids]).sum())         # tổng nat của cả văn bản
    ket_qua.append({"nats": nats, "tokens": len(e.ids), "chars": len(text),
                    "bytes": len(text.encode("utf-8"))})

N = sum(r["nats"] for r in ket_qua)
T = sum(r["tokens"] for r in ket_qua)
C = sum(r["chars"] for r in ket_qua)
B = sum(r["bytes"] for r in ket_qua)
print(f"N = {N:,.0f} nat | T = {T:,} token | C = {C:,} ký tự | B = {B:,} byte")

N = 401,946 nat | T = 54,519 token | C = 208,047 ký tự | B = 274,289 byte


## 8. Mắt xích quan trọng nhất: tổng nat là đại lượng của cả văn bản

Quy tắc chuỗi của xác suất cho mọi phân phối:

$$q(x_{1:T}) = \prod_{t=1}^{T} q(x_t \mid x_{<t})$$

**Ký hiệu mới:** $T$ — số token của văn bản; $x_{1:T}$ — cả chuỗi; $\prod_{t=1}^{T}$ — tích qua mọi vị trí.

Đây là **quy tắc chuỗi** của xác suất, đúng với mọi phân phối chứ không phải một giả định.

Lấy $-\ln$ hai vế, tích thành tổng:

$$N = \sum_{t=1}^{T} \ell_t = -\ln q(x_{1:T})$$

Tức **$N$ không phải đại lượng "theo token"**. Nó là $-\ln$ xác suất mà model gán cho **toàn bộ văn bản**, hay
nói cách khác: số nat cần để mã hoá chính văn bản đó. Cách chia văn bản thành token chỉ là cách *tính* $N$.

Kiểm chứng trực tiếp trên model unigram: nhân xác suất từng token rồi lấy $-\ln$, so với việc cộng từng $-\ln$.

In [14]:
# câu ngắn để nhân trực tiếp được, không bị tràn số dưới
cau = "Hà Nội là thủ đô."
ids = tok.encode(cau, add_special_tokens=False).ids
p_tung_token = q_unigram[ids]

xac_suat_chuoi = float(np.prod(p_tung_token))             # nhân trực tiếp
tong_cac_nat = float(-np.log(p_tung_token).sum())         # cộng -ln từng vị trí

print(f"câu: {cau!r} -> {len(ids)} token")
print(f"xác suất của cả chuỗi (nhân trực tiếp) = {xac_suat_chuoi:.6e}")
print(f"-ln(xác suất chuỗi)                    = {-math.log(xac_suat_chuoi):.6f} nat")
print(f"tổng -ln từng vị trí                   = {tong_cac_nat:.6f} nat      <- bằng nhau")

# với văn bản dài, phép nhân tràn số dưới về 0, nên buộc phải làm việc trong không gian log
ids_dai = tok.encode(eval_docs[0], add_special_tokens=False).ids
print(f"\nvăn bản {len(ids_dai)} token: nhân trực tiếp ra {float(np.prod(q_unigram[ids_dai])):.1f} (tràn số dưới)")
print(f"trong không gian log vẫn tính được: {float(-np.log(q_unigram[ids_dai]).sum()):.1f} nat")

câu: 'Hà Nội là thủ đô.' -> 6 token
xác suất của cả chuỗi (nhân trực tiếp) = 3.040416e-19
-ln(xác suất chuỗi)                    = 42.637122 nat
tổng -ln từng vị trí                   = 42.637122 nat      <- bằng nhau

văn bản 662 token: nhân trực tiếp ra 0.0 (tràn số dưới)
trong không gian log vẫn tính được: 4732.8 nat


## 9. Ba mẫu số, ba chỉ số

Đã có tử số $N$. Chia cho cái gì là lựa chọn của người báo cáo:

$$\text{bit/token} = \frac{N}{T\ln 2}, \qquad \mathrm{bpb} = \frac{N}{B\ln 2}, \qquad \mathrm{bpc} = \frac{N}{C\ln 2}$$

**Ký hiệu mới**
- $B$ — số **byte** UTF-8 của văn bản (đổi theo NFC/NFD)
- $C$ — số **ký tự NFC** của văn bản (chỉ phụ thuộc văn bản)
- bpb = *bits per byte*, bpc = *bits per character*

và perplexity là cách viết khác của bit/token:

$$\mathrm{PPL} = \exp\left(\frac{N}{T}\right)$$

**Diễn giải:** $\mathrm{PPL}$ là số lựa chọn ngang nhau mà model "phân vân" ở mỗi token; model đoán đều trên
$|V|$ token có $\mathrm{PPL} = |V|$.

In [15]:
def cac_chi_so(N, T, B, C):
    return {"bit/token": N / (T * math.log(2)),
            "bpb": N / (B * math.log(2)),
            "bpc": N / (C * math.log(2)),
            "PPL": math.exp(N / T)}

for k, v in cac_chi_so(N, T, B, C).items():
    print(f"{k:10s} = {v:8.4f}")
print("\nmodel thật của bạn ở d8 đạt bpc = 1.0019 — unigram tệ hơn nhiều, đúng như kỳ vọng")

bit/token  =  10.6364
bpb        =   2.1141
bpc        =   2.7873
PPL        = 1591.7521

model thật của bạn ở d8 đạt bpc = 1.0019 — unigram tệ hơn nhiều, đúng như kỳ vọng


Model unigram chỉ đếm tần suất, không nhìn ngữ cảnh, nên bpc của nó cao hơn model thật rất nhiều. Khoảng cách
giữa hai con số chính là phần mà Transformer học được từ ngữ cảnh.

## 10. Vì sao mẫu số phải bất biến

Project so 4 điều kiện khác nhau ở hai chiều: tokenizer và chuẩn hoá Unicode. Nguyên tắc: **mẫu số phải là đại
lượng không đổi khi ta thay đổi thứ đang so sánh**. Nếu mẫu số cũng đổi, ta không biết chênh lệch đến từ tử số
(chất lượng model) hay từ mẫu số (cách đếm).

Cell dưới đo cả ba mẫu số cho cùng một tập văn bản, dưới cả 4 điều kiện.

In [16]:
bang = []
for cond in ["bpe-nfc", "bpe-nfd", "super-nfc", "super-nfd"]:
    t = Tokenizer.from_file(str(DATA / "tokenizers-16k" / cond / "tokenizer.json"))
    T_c = sum(len(e.ids) for e in t.encode_batch(eval_docs, add_special_tokens=False))
    dang = "NFD" if cond.endswith("nfd") else "NFC"
    van_ban = [unicodedata.normalize(dang, d) for d in eval_docs]
    bang.append({"điều kiện": cond,
                 "T (token)": T_c,
                 "B (byte)": sum(len(v.encode()) for v in van_ban),
                 "C (ký tự NFC)": sum(len(d) for d in eval_docs)})

print(f"{'điều kiện':12s} {'T (token)':>12s} {'B (byte)':>12s} {'C (ký tự NFC)':>15s}")
for r in bang:
    print(f"{r['điều kiện']:12s} {r['T (token)']:12,d} {r['B (byte)']:12,d} {r['C (ký tự NFC)']:15,d}")

điều kiện       T (token)     B (byte)   C (ký tự NFC)
bpe-nfc            54,519      274,289         208,047
bpe-nfd            54,556      323,629         208,047
super-nfc          44,811      274,289         208,047
super-nfd          44,857      323,629         208,047


Đọc bảng theo cột:

- **T đổi theo tokenizer**: SuperBPE ít hơn khoảng 18%. Dùng T làm mẫu số thì SuperBPE bị phạt oan.
- **B đổi theo chuẩn hoá**: NFD nhiều hơn khoảng 18%. Dùng B làm mẫu số thì NFD được thưởng oan.
- **C không đổi**: số ký tự NFC là thuộc tính của văn bản gốc.

Nên chỉ có bpc so sánh được. Đây là toàn bộ lý do của dòng bất biến trong `CLAUDE.md`.

## 11. Hai minh chứng trên số liệu thật của project

### 11.1. bpb thiên vị NFD, đo được chính xác

Nếu hai model chất lượng y hệt nhau (cùng $N$), tỷ lệ bpb chỉ còn là nghịch đảo tỷ lệ byte:

$$\frac{\mathrm{bpb}_{\text{nfd}}}{\mathrm{bpb}_{\text{nfc}}} = \frac{N/B_{\text{nfd}}}{N/B_{\text{nfc}}} = \frac{B_{\text{nfc}}}{B_{\text{nfd}}}$$

In [17]:
nfc_all = "".join(docs)
nfd_all = unicodedata.normalize("NFD", nfc_all)
B_nfc, B_nfd = len(nfc_all.encode()), len(nfd_all.encode())

print(f"toàn bộ test.jsonl: NFC {B_nfc:,} byte | NFD {B_nfd:,} byte  (NFD nhiều hơn {B_nfd/B_nfc - 1:.1%})")
print(f"dự đoán tỷ lệ bpb nếu chất lượng như nhau = {B_nfc / B_nfd:.4f}")

# số thật lấy từ log huấn luyện d8
bpb_nfc, bpb_nfd = 0.719967, 0.610563
print(f"tỷ lệ bpb quan sát được trong log         = {bpb_nfd / bpb_nfc:.4f}")
print("\n-> hai tỷ lệ chỉ lệch nhau ~0,0006: gần như toàn bộ 15% khoảng cách là hiện tượng đếm mẫu số,")
print("   không phải chất lượng model. (Hai số không trùng tuyệt đối vì bpb trong log đo trên shard val,")
print("   còn tỷ lệ byte ở đây đo trên test.jsonl, và hai model cũng không có N giống hệt nhau.)")

toàn bộ test.jsonl: NFC 5,523,504 byte | NFD 6,518,395 byte  (NFD nhiều hơn 18.0%)
dự đoán tỷ lệ bpb nếu chất lượng như nhau = 0.8474
tỷ lệ bpb quan sát được trong log         = 0.8480

-> hai tỷ lệ chỉ lệch nhau ~0,0006: gần như toàn bộ 15% khoảng cách là hiện tượng đếm mẫu số,
   không phải chất lượng model. (Hai số không trùng tuyệt đối vì bpb trong log đo trên shard val,
   còn tỷ lệ byte ở đây đo trên test.jsonl, và hai model cũng không có N giống hệt nhau.)


In [18]:
# vì sao NFD nhiều byte hơn: xem một chữ
for dang in ("NFC", "NFD"):
    s = unicodedata.normalize(dang, "ộ")
    ten = [unicodedata.name(c) for c in s]
    print(f"{dang}: {len(s)} ký tự, {len(s.encode()):>1} byte | {ten}")

NFC: 1 ký tự, 3 byte | ['LATIN SMALL LETTER O WITH CIRCUMFLEX AND DOT BELOW']
NFD: 3 ký tự, 5 byte | ['LATIN SMALL LETTER O', 'COMBINING DOT BELOW', 'COMBINING CIRCUMFLEX ACCENT']


### 11.2. Perplexity không so được giữa hai tokenizer

Quy đổi từ bpc thật của project và số ký tự trên token: $\text{bit/token} = \mathrm{bpc} \times C/T$, rồi
$\mathrm{PPL} = \exp(\text{bit/token} \times \ln 2)$.

($C/T$ là số ký tự mỗi token — cột `chars_per_token` trong `compression-16k.json`.)

In [19]:
comp = json.loads((DATA / "compression-16k.json").read_text())
bpc_d8 = {"bpe-nfc": 1.0019, "super-nfc": 1.0037}          # từ results/summary.md

print(f"{'điều kiện':12s} {'bpc':>8s} {'C/T':>8s} {'bit/token':>11s} {'PPL':>8s}")
ppl = {}
for cond, b in bpc_d8.items():
    cpt = comp[cond]["chars_per_token"]
    bit_token = b * cpt
    ppl[cond] = math.exp(bit_token * math.log(2))
    print(f"{cond:12s} {b:8.4f} {cpt:8.3f} {bit_token:11.3f} {ppl[cond]:8.1f}")

print(f"\nchênh lệch theo bpc: {bpc_d8['super-nfc']/bpc_d8['bpe-nfc'] - 1:+.2%}")
print(f"chênh lệch theo PPL: {ppl['super-nfc']/ppl['bpe-nfc'] - 1:+.0%}   <- sai lệch hoàn toàn")

điều kiện         bpc      C/T   bit/token      PPL
bpe-nfc        1.0019    3.861       3.868     14.6
super-nfc      1.0037    4.694       4.711     26.2

chênh lệch theo bpc: +0.18%
chênh lệch theo PPL: +79%   <- sai lệch hoàn toàn


Hai model chênh nhau 0,18% theo bpc nhưng 79% theo perplexity. Nguyên nhân: token của SuperBPE gánh nhiều chữ
hơn nên khó đoán hơn — đó là tính chất của tokenizer, không phải chất lượng model.

Cùng lý do, `loss` trong log cũng không so được giữa hai điều kiện: 2,615 nat với `bpe-nfc` và 3,195 nat với
`super-nfc`.

## 12. Đọc lại con số bpc của chính bạn

In [20]:
bpc_theo_co = {"d6": 1.0914, "d8": 1.0019, "d10": 0.9370}   # results/summary.md, điều kiện bpe-nfc
byte_moi_ky_tu = B_nfc / len(nfc_all)

print(f"tiếng Việt NFC: {byte_moi_ky_tu:.3f} byte mỗi ký tự khi lưu thô bằng UTF-8\n")
for ten, b in bpc_theo_co.items():
    byte_nen = b / 8
    print(f"{ten}: {b:.4f} bit/ký tự -> lưu 1.000 ký tự hết {b*1000/8:.0f} byte "
          f"(thô: {byte_moi_ky_tu*1000:.0f} byte, nén {byte_moi_ky_tu/byte_nen:.1f} lần)")

tiếng Việt NFC: 1.317 byte mỗi ký tự khi lưu thô bằng UTF-8

d6: 1.0914 bit/ký tự -> lưu 1.000 ký tự hết 136 byte (thô: 1317 byte, nén 9.7 lần)
d8: 1.0019 bit/ký tự -> lưu 1.000 ký tự hết 125 byte (thô: 1317 byte, nén 10.5 lần)
d10: 0.9370 bit/ký tự -> lưu 1.000 ký tự hết 117 byte (thô: 1317 byte, nén 11.2 lần)


Để có một mốc về độ lớn: Shannon (1951) ước lượng entropy của tiếng Anh viết vào khoảng 0,6–1,3 bit mỗi ký
tự bằng thực nghiệm đoán chữ với người (27 ký hiệu: chữ cái và dấu cách). Con số 0,94–1,09 của bạn nằm cùng
thang. Đây chỉ là kiểm tra độ lớn, **không** phải phép so sánh: khác ngôn ngữ, khác bảng ký hiệu (văn bản web
của project có chữ hoa, số, dấu câu, dấu thanh), và model người trong thí nghiệm của Shannon mạnh hơn nhiều.

## 13. Một điều kiện cần mà dễ bị bỏ qua

Cả lập luận "bpc là số bit của văn bản" chỉ đúng nếu chuỗi token **xác định lại được đúng văn bản**. Nếu
tokenizer làm mất thông tin, $-\ln q(\text{chuỗi token})$ không còn là số bit của văn bản — và tệ hơn, một
tokenizer mất nhiều thông tin sẽ cho bpc **thấp** một cách giả tạo.

Đó là ý nghĩa thật của phép kiểm tra round-trip trong notebook Kaggle `kaggle/notebooks/01_data_tokenizers.ipynb`
(cell "Step 3 checks": mã hoá rồi giải mã 1.000 văn bản val cho mọi tokenizer, `assert bad == 0`), không chỉ là
kiểm tra cho chắc. Cell dưới lặp lại phép kiểm tra đó trên văn bản test.

In [21]:
loi = 0
for cond in ["bpe-nfc", "bpe-nfd", "super-nfc", "super-nfd"]:
    t = Tokenizer.from_file(str(DATA / "tokenizers-16k" / cond / "tokenizer.json"))
    for d, e in zip(eval_docs, t.encode_batch(eval_docs, add_special_tokens=False)):
        if unicodedata.normalize("NFC", t.decode(e.ids)) != d:
            loi += 1
print(f"số văn bản không khôi phục được: {loi} / {len(eval_docs) * 4}")

số văn bản không khôi phục được: 0 / 400


## 14. Bài tập tự kiểm

Làm trong cell dưới, đáp án ở cell cuối.

1. Model gán xác suất $0{,}02$ cho token thật ở một vị trí. Vị trí đó đóng góp bao nhiêu nat, bao nhiêu bit?
2. Một model đoán đều trên vocab 16.009 với tokenizer đạt 3,861 ký tự mỗi token thì bpc bằng bao nhiêu?
3. Model A có $\mathrm{PPL} = 14{,}6$ và $C/T = 3{,}861$; model B có $\mathrm{PPL} = 26{,}2$ và $C/T = 4{,}694$.
   Model nào nén tốt hơn? Tính ra con số.
4. Nếu ai đó báo cáo bpb và kết luận NFD tốt hơn 15%, bạn cần đo thêm đại lượng nào để phản biện?

In [22]:
# chỗ làm bài

In [23]:
# đáp án
print("1.", f"{-math.log(0.02):.2f} nat = {-math.log2(0.02):.2f} bit")
print("2.", f"{math.log2(16009) / 3.861:.2f} bit mỗi ký tự")
print("3.", f"A: {math.log2(14.6)/3.861:.3f} bpc | B: {math.log2(26.2)/4.694:.3f} bpc -> gần như ngang nhau")
print("4. số byte của cùng tập văn bản ở hai dạng chuẩn hoá; nếu tỷ lệ bpb trùng B_nfc/B_nfd")
print("   thì toàn bộ khác biệt là hiện tượng đếm mẫu số")

1. 3.91 nat = 5.64 bit
2. 3.62 bit mỗi ký tự
3. A: 1.002 bpc | B: 1.004 bpc -> gần như ngang nhau
4. số byte của cùng tập văn bản ở hai dạng chuẩn hoá; nếu tỷ lệ bpb trùng B_nfc/B_nfd
   thì toàn bộ khác biệt là hiện tượng đếm mẫu số


## 14b. Bốn cái bẫy

1. **Đừng lấy trung bình của các tỷ lệ.** $\frac{1}{n}\sum_i \frac{N_i}{C_i}$ khác $\frac{\sum_i N_i}{\sum_i C_i}$.
   ($N_i$, $C_i$: tổng nat và số ký tự NFC của văn bản thứ $i$, $n$ văn bản.) Dạng thứ nhất cho mỗi văn bản một
   phiếu như nhau.
   `vitok.stats.bpc` dùng dạng thứ hai — tổng chia tổng — nên văn bản dài có trọng số lớn hơn.
2. **Đừng so hai chỉ số đo trên hai tập văn bản khác nhau.** bpb của nanochat đo trên shard val, bpc của bạn đo
   trên `test.jsonl`; chỉ so tỷ lệ trong cùng một cột.
3. **Ký tự NFC, không phải ký tự NFD.** Cùng tập test có 4.192.698 ký tự NFC và 5.307.420 ký tự NFD; dùng nhầm
   mẫu số thứ hai thì bpc bị chia cho số lớn hơn 26,6%.
4. **Tổng nat là chặn trên.** Model gán xác suất cho *chuỗi token chuẩn tắc* mà tokenizer sinh ra, trong khi
   cùng một văn bản về nguyên tắc tách được thành nhiều chuỗi khác; xác suất đầy đủ của văn bản là tổng trên
   mọi cách tách. Sai khác nhỏ, có ở mọi công trình dùng chỉ số kiểu này, nhưng nên ghi vào phần hạn chế.

In [24]:
# bẫy 1 trên dữ liệu thật: hai cách tính chênh nhau bao nhiêu
tong_chia_tong = N / (C * math.log(2))
trung_binh_ty_le = float(np.mean([r["nats"] / (r["chars"] * math.log(2)) for r in ket_qua]))
print(f"tổng chia tổng       = {tong_chia_tong:.4f} bpc   <- vitok.stats.bpc")
print(f"trung bình các tỷ lệ = {trung_binh_ty_le:.4f} bpc")
print(f"chênh lệch           = {abs(tong_chia_tong - trung_binh_ty_le) / tong_chia_tong:.2%}")

tổng chia tổng       = 2.7873 bpc   <- vitok.stats.bpc
trung bình các tỷ lệ = 2.8018 bpc
chênh lệch           = 0.52%


## 15. Tóm tắt

| Khái niệm | Công thức | Vai trò trong project |
|---|---|---|
| Thông tin của một sự kiện | $I(p) = -\log p$ | Nền của mọi thứ còn lại |
| Entropy | $H(p) = -\sum p\log_2 p$ | Mốc dưới: không model nào đi dưới mức này |
| Cross-entropy | $H(p,q) = -\sum p\log_2 q$ | Chính là hàm loss khi train |
| KL divergence | $H(p,q) - H(p)$ | Phần phạt do model chưa đúng; hiệu cross-entropy giữa hai model bằng hiệu KL |
| Tổng nat của văn bản | $N = -\ln q(x_{1:T})$ | Tử số bất biến, không phụ thuộc cách chia token |
| bpc | $N / (C\ln 2)$ | **Chỉ số chính thức của project** |
| bpb | $N / (B\ln 2)$ | Thiên vị NFD 15%, không dùng |
| Perplexity | $\exp(N/T)$ | Thiên vị tokenizer 79%, không dùng để so |

Tiếp theo: notebook 02 về thống kê suy diễn — khoảng tin cậy, bootstrap, nhiễu seed.

**Nguồn đọc thêm**

- [Jurafsky & Martin, SLP3 chương 3](https://web.stanford.edu/~jurafsky/slp3/3.pdf) — "Perplexity" và
  "Entropy, Cross-Entropy", cùng nội dung, có thêm ví dụ n-gram.
- [Benchmarking BPE Tokenizers with Bits per Byte](https://aclanthology.org/2026.mellm-1.27/) — cái bẫy bpb.
- Shannon, *Prediction and Entropy of Printed English* (1951) — nguồn của mốc 0,6–1,3 bit mỗi ký tự.